# Synthetic (SMT) sweep analysis

Analyses the synthetic sweep (`cluster/`): a multi-head MLP trained on datasets of
increasing difficulty, with and without the learned-abstention head. The focus is the
abstention story -- why harder datasets are harder, and how much abstaining buys back.

Paper-ready figures are written to `figs/fig_*.{png,pdf}` as they render:

| figure | what it shows |
|---|---|
| `fig_feature_separability_<dataset>` | per-feature class-conditional densities + overlap, baseline vs hardest |
| `fig_risk_coverage` | risk vs coverage: abstention's coverage/accuracy trade-off across `o` |
| `fig_selective_vs_o` | selective accuracy (kept rows) vs `o` on `balanced_hard` |
| `fig_cascade_vs_abstention_selective` | cascade full-coverage vs abstention selective accuracy, per dataset |
| `fig_selective_risk` | selective-risk curves per dataset (easy -> hard) vs confidence / CE / Bayes |
| `fig_selective_accuracy` | selective-accuracy curves per dataset (= 1 - the risk plot) |

Prereq: run the sweep first (`bash cluster/submit.sh` on SLURM, or
`bash cluster/run_local.sh` locally), then run this notebook from the repo root.


## Load every run

Loads the manifest + each run's metrics into one dataframe, with helpers to slice by study and average over seeds.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd

# find the repo root by walking up until cluster/manifest.json appears
root = Path.cwd()
while not (root / "cluster" / "manifest.json").exists() and root != root.parent:
    root = root.parent
manifest = json.loads((root / "cluster" / "manifest.json").read_text())
print("runs in matrix:", len(manifest["runs"]))


def load_rows():
    """One flat row per run that produced a metrics file. Hyperparameters + study
    tags come from the manifest; scores from the run's metrics JSON."""
    rows, missing = [], []
    for r in manifest["runs"]:
        p = root / r["metrics"]
        if not p.exists():
            missing.append(r["run_id"]); continue
        m = json.loads(p.read_text())
        d, mech = m["defect_head"], m["mechanism_accuracy"]
        row = {
            "run_id": r["run_id"], "dataset": r["dataset"], "loss": r["loss"],
            "o": r["o"], "seed": r["seed"], "hidden": r.get("hidden"),
            "dropout": r.get("dropout"), "class_weight": r.get("class_weight"),
            "lr": r.get("lr"), "epochs": r.get("epochs"),
            "studies": r.get("studies", []),
            "defect_acc": d["accuracy"], "defect_wf1": d["weighted_f1"],
            "defect_macrof1": d["macro_f1"],
            "bayes": m["bayes_optimal_accuracy"],
            "gap_to_bayes": m["bayes_optimal_accuracy"] - d["accuracy"],
            "mech_joint": mech["joint_accuracy"],
            "risk_mae": m["risk_mae"],
        }
        ab = m.get("abstention")
        if ab:
            row["mean_abstain"] = ab["mean_abstain_prob"]
            half = ab["by_threshold"].get("0.5", {})
            row["coverage@0.5"] = half.get("coverage")
            row["selective_acc@0.5"] = half.get("selective_accuracy")
        rows.append(row)
    if missing:
        print(f"WARNING: {len(missing)} runs have no metrics yet (still training?):")
        print("  " + ", ".join(missing[:12]) + (" ..." if len(missing) > 12 else ""))
    return pd.DataFrame(rows)


df = load_rows()
print(f"loaded {len(df)} of {len(manifest['runs'])} runs")


def study(name):
    """Rows tagged with a study (core, payoff, capacity, dropout, lr, class_weight)."""
    if not len(df) or "studies" not in df.columns:
        return df.iloc[0:0]
    return df[df["studies"].apply(lambda s: name in (s or []))]


def seed_mean(frame, keys, metrics):
    """Average over seeds -> one row per config (keeps NaN-keyed groups, e.g. o=None)."""
    metrics = [m for m in metrics if m in frame.columns]
    return frame.groupby(keys, dropna=False)[metrics].mean().reset_index()


def core_grid():
    """(core rows, dataset order, losses, bar x-positions, bar width)."""
    core = study("core")
    datasets = list(manifest["datasets"])
    losses = [l for l in ["cascade", "abstention"] if l in set(core["loss"])] if len(core) else []
    x = np.arange(len(datasets)); w = 0.8 / max(len(losses), 1)
    return core, datasets, losses, x, w

## Plot style

Shared paper style + a `save_fig` helper that writes `figs/fig_*.{png,pdf}`.

In [ ]:
# Shared paper-quality plotting style + a figs/ saver. Consistent colors for the two
# losses everywhere; horizontal layouts + zoomed axes so small differences are legible.
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 220, "savefig.bbox": "tight",
    "font.size": 12, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 12, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
})
COLORS = {"cascade": "#2c7fb8", "abstention": "#e6550d", "bayes": "#333333"}

FIGDIR = root / "figs"
FIGDIR.mkdir(exist_ok=True)


def save_fig(fig, name):
    """Write a paper copy (PNG + PDF) to figs/ and print the path."""
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}")
    print(f"saved figs/{name}.png (+ .pdf)")

## Why the hardest problem is harder (feature separability)

Per-feature class-conditional densities for the **baseline** vs the **hardest** dataset -- visually compare the increased overlap. Low open/bridge overlap = the feature separates the defects; high overlap = it carries little signal, which is what makes the task hard. **Saved: `fig_feature_separability_<dataset>`.**

In [ ]:
# FEATURE SEPARABILITY: per-feature class-conditional densities, drawn for the BASELINE
# (well-separated) and the HARDEST dataset (more overlap) so you can VISUALLY compare the
# increased overlap. LOW open/bridge overlap => the feature separates the defects; HIGH
# overlap => it carries little signal, which is what makes the task hard. Dashed lines are
# the spec limits (lsl / usl). Reads only the dataset CSVs + the spec YAML.
csvs = {d: root / "data" / "cluster" / f"{d}.csv" for d in manifest["datasets"]}
csvs = {d: p for d, p in csvs.items() if p.exists()}
if csvs:
    def _berr(p):
        dd = pd.read_csv(p, usecols=lambda c: c.startswith("p_"))
        return float((1 - dd.to_numpy().max(1)).mean())

    # spec limits (lsl/usl) + the monitored feature ids from the YAML (no pyyaml dep)
    lims, cur, in_p = {}, None, False
    for line in (root / "domain" / "smt_paper.yaml").read_text().splitlines():
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        if not line[0].isspace():
            in_p = line.strip().startswith("parameters:"); cur = None; continue
        if not in_p:
            continue
        s = line.strip()
        if s.startswith("- id:"):
            cur = s.split("id:", 1)[1].split("#")[0].strip(); lims[cur] = []
        elif cur and (s.startswith("lsl:") or s.startswith("usl:")):
            lims[cur].append(float(s.split(":", 1)[1].split("#")[0]))
    CLS = [("no_defect", "no defect", "#9e9e9e"),
           ("open_circuit", "open circuit", COLORS["cascade"]),
           ("solder_bridging", "solder bridging", COLORS["abstention"])]

    def featsep(ds):
        dh = pd.read_csv(csvs[ds])
        feats = [f for f in lims if f in dh.columns]     # exactly the process features, spec order
        ncol = 3; nrow = int(np.ceil(len(feats) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.6 * nrow), squeeze=False)
        for k, f in enumerate(feats):
            ax = axes[k // ncol][k % ncol]
            bins = np.linspace(dh[f].min(), dh[f].max(), 60)
            hists = {}
            for key, lab, col in CLS:
                v = dh.loc[dh["defect_label"] == key, f].to_numpy()
                if len(v):
                    ax.hist(v, bins=bins, density=True, alpha=0.55, color=col, label=lab)
                    h, _ = np.histogram(v, bins=bins); hists[key] = h / max(h.sum(), 1)
            if "open_circuit" in hists and "solder_bridging" in hists:
                ov = float(np.minimum(hists["open_circuit"], hists["solder_bridging"]).sum())
            else:
                ov = float("nan")
            for lim in lims.get(f, []):
                ax.axvline(lim, ls="--", color="k", lw=1, alpha=0.7)
            ax.set_title(f"{f}\nopen/bridge overlap = {ov:.2f}", fontsize=10); ax.grid(alpha=0.2)
        for k in range(len(feats), nrow * ncol):
            axes[k // ncol][k % ncol].axis("off")
        axes[0][0].legend(fontsize=9)
        fig.suptitle(f"Feature values by defect class (density) -- {ds} (Bayes err {_berr(csvs[ds]):.3f})"
                     "  --  low overlap = the class colors separate", fontweight="bold")
        fig.tight_layout(); save_fig(fig, f"fig_feature_separability_{ds}"); plt.show()

    hardest = max(csvs, key=lambda d: _berr(csvs[d]))
    baseline = "baseline" if "baseline" in csvs else min(csvs, key=lambda d: _berr(csvs[d]))
    for ds in dict.fromkeys([baseline, hardest]):        # baseline first, dedup if same
        featsep(ds)
else:
    print("no dataset CSVs found (need data/cluster/*.csv on the cluster)")

## Risk vs coverage

The selective-classification figure: coverage vs selective accuracy as the payoff `o` sweeps (one operating point per model); star = cascade at full coverage. **Saved: `fig_risk_coverage`.**

In [ ]:
# PAPER FIGURE: risk-coverage curve. Each abstention model (one per payoff o) gives
# one (coverage, selective-accuracy) point at the h=0.5 accept threshold; sweeping
# o = 1 -> 4 traces the frontier. The star marks cascade at full coverage (it never
# abstains). Points up-and-left of a star => abstaining buys accuracy on the boards
# the model chooses to answer.
RC_DATASETS = ["baseline", "hard", "imbalanced"]      # the original three
pay = study("payoff"); core = study("core")
if len(pay):
    sm = seed_mean(pay, ["dataset", "o"], ["coverage@0.5", "selective_acc@0.5"])
    cmap = plt.get_cmap("tab10")
    fig, ax = plt.subplots(figsize=(8.5, 6))
    for j, d in enumerate([d for d in RC_DATASETS if d in set(pay["dataset"])]):
        s = sm[sm.dataset == d].sort_values("coverage@0.5")
        ax.plot(s["coverage@0.5"], s["selective_acc@0.5"], "-o", ms=4,
                color=cmap(j), label=d)
        cba = core[(core.dataset == d) & (core.loss == "cascade")]["defect_acc"].mean()
        ax.scatter(1.0, cba, marker="*", s=190, color=cmap(j),
                   edgecolor="k", linewidth=0.6, zorder=5)
    ax.set_xlabel("coverage  (fraction of boards the model answers)")
    ax.set_ylabel("selective accuracy  (on the answered boards)")
    ax.set_title("Risk-coverage: abstention trades coverage for accuracy\n"
                 "line = abstention swept over o;   star = cascade at full coverage")
    ax.legend(title="dataset", loc="lower left")
    save_fig(fig, "fig_risk_coverage"); plt.show()
else:
    print("no payoff-study runs with metrics yet")

## Selective-classification analysis (threshold-swept)

Loads the saved checkpoints and sweeps the **rejection threshold** on each model (the standard selective-risk view). Runs on the cluster (needs `results/cluster/*.pt` + `data/cluster/*.csv`); builds `sel_table` + the `_find/_infer/_risk_cov` helpers used below.

In [ ]:
# Selective-classification analysis (replicates the toy_example). Unlike the cells
# above (which read only the JSON metrics), this loads the SAVED abstention checkpoints and
# sweeps coverage on ONE model -- the learned reject head vs the same model's softmax
# confidence, both referenced to that model's own full-coverage (CE) error.
# Needs results/cluster/*.pt + data/cluster/*.csv, so run it on the cluster (or after
# rsync-ing them back). Edit SEL_O / SEL_SEED / SEL_COVERAGE to probe other settings.
import sys
sys.path.insert(0, str(root))            # so `import mlp` works regardless of cwd
import torch
import mlp

SEL_O, SEL_SEED, SEL_COVERAGE = 2.0, 0, 0.80
_spec = mlp.load_spec(str(root / "domain" / "smt_paper.yaml"))
_names = mlp.defect_names(_spec)
_ids = mlp.param_ids(_spec)
_covs = np.linspace(0.05, 1.0, 40)


def _find(dataset, loss, o, seed):
    for r in manifest["runs"]:
        if (r["dataset"] == dataset and r["loss"] == loss and r["seed"] == seed
                and (r["o"] == o if loss == "abstention" else True)):
            return r
    return None


def _infer(run, dfd):
    """Test-split predictions for one checkpoint (standardized with its stored stats)."""
    model, ckpt = mlp.load_model(str(root / run["model"]), device="cpu")
    te = np.where(dfd["split"].to_numpy() == "test")[0]
    mu, sd = ckpt["standardize"]["mu"], ckpt["standardize"]["sd"]
    X = ((dfd.iloc[te][_ids].to_numpy(np.float32) - mu) / sd).astype(np.float32)
    y = dfd.iloc[te]["defect_label"].map({n: i for i, n in enumerate(_names)}).to_numpy()
    p = model.predict(torch.from_numpy(X))
    d = {"y": y, "argmax": p["defect_argmax"].cpu().numpy(),
         "real": p["defect_prob"].cpu().numpy(), "te": te,
         "abstain": bool(getattr(model, "abstain", False))}
    if d["abstain"]:
        d["r"] = p["abstain_prob"].cpu().numpy()
    return d


def _risk_cov(score, correct, covs=_covs):
    """Rank-based selective risk: accept the most-confident `cov` fraction (highest
    score first) -> accepted error at each coverage."""
    order = np.argsort(-score); n = len(score)
    return np.array([1 - correct[order[:max(1, int(round(c * n)))]].mean() for c in covs])


def _bayes_err(dfd, te):
    post = dfd.iloc[te][[f"p_{n}" for n in _names]].to_numpy()
    return float((1 - post.max(1)).mean())


SEL_SEEDS = sorted({r["seed"] for r in manifest["runs"]})   # every seed -> mean curve + seed band

sel, _rows = {}, []
for ds in manifest["datasets"]:
    csv = root / "data" / "cluster" / f"{ds}.csv"
    if not csv.exists():
        continue
    dfd = None
    abst_stack, conf_stack, ce_list = [], [], []      # one curve/value per seed
    bayes, ia_last = None, None
    for seed in SEL_SEEDS:
        abst = _find(ds, "abstention", SEL_O, seed)   # ONE model for all three curves
        if not (abst and (root / abst["model"]).exists()):
            continue
        if dfd is None:
            dfd = pd.read_csv(csv)
        ia = _infer(abst, dfd)
        correct = (ia["argmax"] == ia["y"]).astype(float)
        ce_list.append(1 - correct.mean())                            # THIS model's full-coverage error
        abst_stack.append(_risk_cov(-ia["r"], correct))               # rank by the learned reject head
        conf_stack.append(_risk_cov(ia["real"].max(1), correct))      # rank by the SAME model's softmax confidence
        if bayes is None:
            bayes = _bayes_err(dfd, ia["te"])         # dataset property -- same across seeds
        ia_last = ia
    if not abst_stack:
        continue
    abst_stack, conf_stack, ce_arr = np.array(abst_stack), np.array(conf_stack), np.array(ce_list)
    abst_err, conf_err, ce_err = abst_stack.mean(0), conf_stack.mean(0), float(ce_arr.mean())
    sel[ds] = dict(bayes=bayes, ce_err=ce_err, abst_err=abst_err, conf_err=conf_err,
                   abst_err_seeds=abst_stack, conf_err_seeds=conf_stack,   # [n_seeds, n_covs]
                   ce_err_seeds=ce_arr, n_seeds=len(abst_stack), ia=ia_last)
    _at = lambda e: float(e[int(np.argmin(np.abs(_covs - SEL_COVERAGE)))])
    _rows.append(dict(dataset=ds, bayes_err=bayes, ce_full_err=ce_err, n_seeds=len(abst_stack),
                      abst_err_at=_at(abst_err), conf_err_at=_at(conf_err),
                      gain_vs_ce=ce_err - _at(abst_err),
                      gain_vs_conf=_at(conf_err) - _at(abst_err)))

if _rows:
    sel_table = pd.DataFrame(_rows).sort_values("bayes_err").reset_index(drop=True)
    print(f"selective analysis: {len(sel)} datasets  (abstention o={SEL_O:g}, mean over seeds "
          f"{SEL_SEEDS}, reported @ coverage~{SEL_COVERAGE:.2f})")
    print(sel_table.round(4).to_string(index=False))
    print("\ngain_vs_ce   = full-coverage error - abstention accepted error   (>0: abstaining helps vs never rejecting)")
    print("gain_vs_conf = softmax-confidence error - reject-head error   (>0: the LEARNED reject beats the model's own softmax confidence)")
else:
    sel_table = pd.DataFrame()
    print("No checkpoints found -- run this on the cluster (needs results/cluster/*.pt + data/cluster/*.csv).")

## Selective accuracy vs the payoff o

Selective accuracy (kept rows) vs the payoff `o` on `balanced_hard`, from each `o`'s checkpoint. It rises to an interior peak, then comes back down as the model stops abstaining. **Saved: `fig_selective_vs_o`.**

In [ ]:
# Selective accuracy vs the payoff o, on balanced_hard (the dataset that under-fits at low o).
# Each o's abstention model operates at its own reject decision but never below a coverage FLOOR;
# selective accuracy is the accuracy on the kept rows. As o rises the model abstains less and the
# curve settles onto the plain classifier -- so it rises to an interior peak, then comes back down.
# Needs results/cluster/*.pt + data/cluster/*.csv.
SEL_DS = "balanced_hard"
FLOOR = 0.60
SMOOTH_WIN = 3                # rolling-mean window for display smoothing (1 = raw, no smoothing)
pay_runs_all = [r for r in manifest["runs"] if "payoff" in (r.get("studies") or [])]
SELVO_SEEDS = sorted({r["seed"] for r in pay_runs_all})

pay_ds = sorted({r["dataset"] for r in pay_runs_all})
if SEL_DS not in pay_ds:      # fall back to the hardest-by-Bayes payoff dataset if not present
    def _bayes_acc(d):
        for r in pay_runs_all:
            if r["dataset"] == d and (root / r["metrics"]).exists():
                return json.loads((root / r["metrics"]).read_text()).get("bayes_optimal_accuracy", 1.0)
        return 1.0
    SEL_DS = min(pay_ds, key=_bayes_acc)
ds = SEL_DS
o_grid = sorted({r["o"] for r in pay_runs_all if r["loss"] == "abstention" and r["dataset"] == ds})

rows = []
csv = root / "data" / "cluster" / f"{ds}.csv"
if csv.exists():
    dfd = pd.read_csv(csv)
    for o in o_grid:
        for seed in SELVO_SEEDS:
            run = _find(ds, "abstention", o, seed)
            if not (run and (root / run["model"]).exists()):
                continue
            info = _infer(run, dfd)
            if not info["abstain"]:
                continue
            correct = (info["argmax"] == info["y"]).astype(float)
            acc = 1 - _risk_cov(-info["r"], correct)
            op_cov = max(float((info["r"] < 0.5).mean()), FLOOR)   # own coverage, never below the floor
            rows.append({"o": o, "sel": float(np.interp(op_cov, _covs, acc))})
selvo = pd.DataFrame(rows)
if len(selvo):
    sm = selvo.groupby("o")["sel"].mean().reset_index().sort_values("o").reset_index(drop=True)
    o_arr = sm["o"].to_numpy()
    sel_s = sm["sel"].rolling(SMOOTH_WIN, center=True, min_periods=1).mean().to_numpy()
    peak_i = int(np.nanargmax(sel_s)); peak_o = float(o_arr[peak_i])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(o_arr, sel_s, marker="o", color="#1f77b4", lw=2)
    ax.axvline(peak_o, color="#1f77b4", ls=":", lw=1)
    ax.annotate(f"best o={peak_o:g}", (peak_o, sel_s[peak_i]),
                textcoords="offset points", xytext=(6, 6), color="#1f77b4")
    ax.set_xlabel("payoff  o"); ax.set_ylabel("selective accuracy (kept rows)")
    ax.set_title("Selective accuracy vs o")
    ax.grid(alpha=0.3)
    save_fig(fig, "fig_selective_vs_o"); plt.show()
    print(sm.round(4).to_string(index=False))
else:
    print(f"no abstention checkpoints for {ds}")

## Cascade vs abstention (selective accuracy)

Full-coverage cascade accuracy vs abstention's selective accuracy on the boards it answers (`r<0.5`), per dataset, coverage annotated. **Saved: `fig_cascade_vs_abstention_selective`.**

In [ ]:
core = study("core")
datasets = list(manifest["datasets"])
cas = seed_mean(core[core.loss == "cascade"], ["dataset"], ["defect_acc"])
ab  = seed_mean(core[core.loss == "abstention"], ["dataset"], ["selective_acc@0.5", "coverage@0.5"])

def _g(frame, ds, col):
    v = frame[frame.dataset == ds][col]
    return float(v.iloc[0]) if len(v) else np.nan

order = sorted(datasets, key=lambda d: _g(cas, d, "defect_acc"))
x = np.arange(len(order)); w = 0.38
cas_acc = [_g(cas, d, "defect_acc") for d in order]
sel_acc = [_g(ab, d, "selective_acc@0.5") for d in order]
cov     = [_g(ab, d, "coverage@0.5") for d in order]

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - w/2, cas_acc, w, color=COLORS["cascade"], label="cascade (full coverage)")
ax.bar(x + w/2, sel_acc, w, color=COLORS["abstention"], label="abstention (selective, r<0.5)")
for xi, s, c in zip(x, sel_acc, cov):
    if not np.isnan(s):
        ax.annotate(f"cov {c:.2f}", (xi + w/2, s), ha="center", va="bottom", fontsize=8)
allv = [v for v in cas_acc + sel_acc if not np.isnan(v)]
ax.set_ylim(max(0.0, min(allv) - 0.03), 1.0)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=25, ha="right")
ax.set_ylabel("accuracy")
ax.set_title("Cascade full-coverage accuracy vs abstention selective accuracy (answered boards, r<0.5)")
ax.legend(loc="lower right"); ax.grid(axis="x", alpha=0)
save_fig(fig, "fig_cascade_vs_abstention_selective"); plt.show()

## Comparing the datasets: selective-risk curves

Accepted error vs coverage for every dataset, easy -> hard, vs the confidence baseline / CE / Bayes floor. **Saved: `fig_selective_risk`.**

In [ ]:
# Selective-risk curves (reject threshold swept) per dataset, ordered easy -> hard.
# Single model (the abstention model). Solid/dashed lines are the MEAN over seeds; the
# shaded band spans the per-seed min..max envelope. abstention = rank by the learned reject
# head; softmax confidence = rank the SAME model by its softmax margin. Both start at the
# model's full-coverage error (CE) at coverage 1 and can only fall as coverage drops. Orange
# below blue = the learned reject beats the model's own confidence.
if len(sel):
    order = list(sel_table["dataset"])
    ncol = min(5, len(order)); nrow = int(np.ceil(len(order) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.0 * nrow), squeeze=False)
    for k, ds in enumerate(order):
        ax = axes[k // ncol][k % ncol]; d = sel[ds]
        # mean lines
        ax.plot(_covs, d["abst_err"], "-", color=COLORS["abstention"], lw=2, label="abstention")
        ax.plot(_covs, d["conf_err"], "--", color=COLORS["cascade"], lw=1.8, label="softmax confidence")
        # seed band (min..max over seeds); only when >1 seed is available
        aseeds, cseeds = d.get("abst_err_seeds"), d.get("conf_err_seeds")
        if aseeds is not None and len(aseeds) > 1:
            ax.fill_between(_covs, aseeds.min(0), aseeds.max(0),
                            color=COLORS["abstention"], alpha=0.20, lw=0)
        if cseeds is not None and len(cseeds) > 1:
            ax.fill_between(_covs, cseeds.min(0), cseeds.max(0),
                            color=COLORS["cascade"], alpha=0.15, lw=0)
        ax.axhline(d["ce_err"], color="#999", ls=":", lw=1.4, label="CE full coverage")
        ax.axhline(d["bayes"], color="k", ls="-", lw=1.0, alpha=0.6, label="Bayes floor")
        ax.set_title(f"{ds} (Bayes {d['bayes']:.3f}, {d.get('n_seeds', 1)} seeds)", fontsize=10)
        ax.set_xlabel("coverage"); ax.set_ylabel("accepted error"); ax.grid(alpha=0.25)
    for k in range(len(order), nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    axes[0][0].legend(fontsize=8)
    fig.tight_layout(); save_fig(fig, "fig_selective_risk"); plt.show()
else:
    print("no checkpoints loaded (see the compute cell above)")

## Comparing the datasets: selective-accuracy curves

The same plot with selective accuracy (= 1 - accepted error) on the y-axis: accuracy on the answered boards vs coverage, easy -> hard, vs the confidence baseline / CE full-coverage accuracy / Bayes ceiling. **Saved: `fig_selective_accuracy`.**

In [ ]:
# Selective-ACCURACY curves (reject threshold swept) per dataset, ordered easy -> hard.
# Exactly the risk plot above with accuracy = 1 - accepted error on the y-axis. abstention
# (reject high reservation) vs cascade confidence thresholding vs the CE full-coverage accuracy
# and the Bayes ceiling. Both curves come from the SAME abstention model (reject head vs its
# own softmax confidence) and meet the full-coverage accuracy at coverage 1.
if len(sel):
    order = list(sel_table["dataset"])
    ncol = min(5, len(order)); nrow = int(np.ceil(len(order) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.0 * nrow), squeeze=False)
    for k, ds in enumerate(order):
        ax = axes[k // ncol][k % ncol]; d = sel[ds]
        ax.plot(_covs, 1 - d["abst_err"], "-", color=COLORS["abstention"], lw=2, label="abstention")
        ax.plot(_covs, 1 - d["conf_err"], "--", color=COLORS["cascade"], lw=1.8, label="softmax confidence")
        ax.axhline(1 - d["ce_err"], color="#999", ls=":", lw=1.4, label="CE full coverage")
        ax.axhline(1 - d["bayes"], color="k", ls="-", lw=1.0, alpha=0.6, label="Bayes ceiling")
        ax.set_title(f"{ds} (Bayes {1 - d['bayes']:.3f})", fontsize=10)
        ax.set_xlabel("coverage"); ax.set_ylabel("selective accuracy"); ax.grid(alpha=0.25)
    for k in range(len(order), nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    axes[0][0].legend(fontsize=8)
    fig.tight_layout(); save_fig(fig, "fig_selective_accuracy"); plt.show()
else:
    print("no checkpoints loaded (see the compute cell above)")

## Optimal configurations

In [ ]:
# Optimal configs. core picks are averaged over seeds so we do not chase noise;
# each study reports the best setting of its own axis per probe dataset.
core = study("core")
mean_over_seeds = core.groupby(["dataset", "loss", "o"], dropna=False).mean(numeric_only=True).reset_index()


def best(metric, maximize=True):
    s = mean_over_seeds.sort_values(metric, ascending=not maximize).iloc[0]
    o = "-" if pd.isna(s["o"]) else f"{s['o']:g}"
    return f"{s['dataset']} / {s['loss']} / o={o}  ->  {metric}={s[metric]:.4f}"


print("=== Optimal core configs (mean over seeds) ===")
print("highest defect accuracy :", best("defect_acc"))
print("smallest gap to Bayes   :", best("gap_to_bayes", maximize=False))
print("best minority macro-F1  :", best("defect_macrof1"))
print("highest joint mechanism :", best("mech_joint"))
print("lowest risk MAE         :", best("risk_mae", maximize=False))

print("\n=== Best setting per study (probe datasets, mean over seeds) ===")
for name, axis, metric, maximize in [
        ("capacity", "hidden", "gap_to_bayes", False),
        ("dropout", "dropout", "gap_to_bayes", False),
        ("lr", "lr", "gap_to_bayes", False),
        ("class_weight", "class_weight", "defect_macrof1", True),
        ("payoff", "o", "selective_acc@0.5", True)]:
    s = study(name)
    if not len(s) or metric not in s.columns or not s[metric].notna().any():
        print(f"  {name:12s}: (no runs yet)"); continue
    sm = seed_mean(s, ["dataset", axis], [metric]).dropna(subset=[metric])
    for ds in sorted(sm["dataset"].unique()):
        win = sm[sm.dataset == ds].sort_values(metric, ascending=not maximize).iloc[0]
        print(f"  {name:12s} [{ds:11s}] best {axis}={win[axis]!s:11s} -> {metric}={win[metric]:.4f}")